In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, mannwhitneyu
import matplotlib.pyplot as plt

model_df = pd.read_parquet("../data/model_df_full.parquet")
print(model_df.shape)
model_df.head()

(19368, 27)


,user_id,birth_year,country,city,created_date,user_settings_crypto_unlocked,plan,attributes_notifications_marketing_push,attributes_notifications_marketing_email,num_contacts,...,avg_transaction_amount_obs,active_days_obs,declined_rate_obs,device_brand,num_notifications_obs,failed_notification_rate_obs,num_merchant_categories_obs,num_countries_visited_obs,in_count_obs,out_count_obs
0,user_2811,1988,IE,C,2018-03-04 21:46:39.616834+00:00,0,STANDARD,unknown,unknown,0,...,208.616471,13.0,0.117647,Apple,1.0,1.000000,7.0,2.0,14.0,20.0
1,user_4750,1987,FR,Ry,2018-04-07 06:49:18.345736+00:00,0,STANDARD,unknown,unknown,0,...,4.895000,1.0,0.000000,Android,0.0,0.000000,0.0,0.0,3.0,1.0
2,user_17686,1984,GB,Ayr,2018-12-04 10:28:02.653147+00:00,0,PREMIUM,unknown,unknown,6,...,27.526000,19.0,0.044444,Apple,6.0,0.000000,8.0,2.0,15.0,30.0
3,user_18779,1965,GB,Ayr,2018-12-31 07:42:19.353613+00:00,0,STANDARD,unknown,unknown,0,...,5.500000,1.0,0.000000,Apple,9.0,0.111111,0.0,0.0,2.0,0.0
4,user_7823,1999,PL,Buk,2018-06-07 03:22:40.590209+00:00,0,STANDARD,unknown,unknown,0,...,11.705714,4.0,0.142857,Apple,0.0,0.000000,1.0,1.0,2.0,5.0


In [2]:
def cramers_v(contingency_table):
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    v = np.sqrt((chi2 / n) / min_dim)
    return chi2, p, v


def rank_biserial(group_a, group_b):
    u_stat, p = mannwhitneyu(group_a, group_b, alternative="two-sided")
    n1, n2 = len(group_a), len(group_b)
    r = 1 - (2 * u_stat) / (n1 * n2)
    return u_stat, p, r

In [3]:
categorical_vars = ["plan", "country", "device_brand",
                     "attributes_notifications_marketing_push",
                     "attributes_notifications_marketing_email",
                     "user_settings_crypto_unlocked"]

continuous_vars = ["age", "num_contacts", "total_amount_obs",
                    "avg_transaction_amount_obs", "active_days_obs",
                    "declined_rate_obs", "num_notifications_obs",
                    "failed_notification_rate_obs", "num_merchant_categories_obs",
                    "num_countries_visited_obs", "in_count_obs", "out_count_obs"]

print("=" * 70)
print("CHI-SQUARE (categóricas)")
print("=" * 70)
results_cat = []
for col in categorical_vars:
    table = pd.crosstab(model_df[col], model_df["churn_model"])
    chi2, p, v = cramers_v(table)
    results_cat.append({"variable": col, "chi2": chi2, "p_value": p, "cramers_v": v})
    print(f"{col}: chi2={chi2:.2f}, p={p:.4f}, Cramér's V={v:.4f}")

print("\n" + "=" * 70)
print("MANN-WHITNEY U (continuas)")
print("=" * 70)
results_cont = []
for col in continuous_vars:
    churned = model_df.loc[model_df["churn_model"], col].dropna()
    active = model_df.loc[~model_df["churn_model"], col].dropna()
    u_stat, p, r = rank_biserial(churned, active)
    results_cont.append({"variable": col, "u_stat": u_stat, "p_value": p, "rank_biserial_r": r,
                          "median_churned": churned.median(), "median_active": active.median()})
    print(f"{col}: p={p:.4f}, r={r:.4f} | mediana churned={churned.median():.2f}, activos={active.median():.2f}")

results_df = pd.concat([
    pd.DataFrame(results_cat).assign(effect_size=lambda d: d["cramers_v"]),
    pd.DataFrame(results_cont).assign(effect_size=lambda d: d["rank_biserial_r"].abs())
], ignore_index=True).sort_values("effect_size", ascending=False)

print("\n" + "=" * 70)
print("RANKING POR EFFECT SIZE (mayor a menor)")
print("=" * 70)
print(results_df[["variable", "effect_size", "p_value"]].to_string(index=False))

CHI-SQUARE (categóricas)
plan: chi2=281.04, p=0.0000, Cramér's V=0.1205
country: chi2=313.93, p=0.0000, Cramér's V=0.1273
device_brand: chi2=14.18, p=0.0008, Cramér's V=0.0271
attributes_notifications_marketing_push: chi2=77.17, p=0.0000, Cramér's V=0.0631
attributes_notifications_marketing_email: chi2=24.20, p=0.0000, Cramér's V=0.0353
user_settings_crypto_unlocked: chi2=309.20, p=0.0000, Cramér's V=0.1264

MANN-WHITNEY U (continuas)
age: p=0.0000, r=-0.0511 | mediana churned=41.00, activos=39.00
num_contacts: p=0.0000, r=0.4571 | mediana churned=0.00, activos=7.00
total_amount_obs: p=0.0000, r=0.4647 | mediana churned=18.08, activos=439.01
avg_transaction_amount_obs: p=0.0000, r=0.3392 | mediana churned=5.96, activos=21.70
active_days_obs: p=0.0000, r=0.5336 | mediana churned=1.00, activos=7.00
declined_rate_obs: p=0.0000, r=0.2836 | mediana churned=0.00, activos=0.00
num_notifications_obs: p=0.0000, r=-0.1299 | mediana churned=2.00, activos=2.00
failed_notification_rate_obs: p=0.098

## Revisando multicolinealidad antes de definir el feature set final

Encontré que 6 de las 7 variables con mayor effect size (active_days_obs,
out_count_obs, num_merchant_categories_obs, total_amount_obs,
num_countries_visited_obs, in_count_obs) probablemente midan lo mismo desde
ángulos distintos: un usuario con más transacciones naturalmente toca más
categorías y países, no porque sea "más diverso", sino por simple exposición
(más intentos, más variedad por azar). Antes de decidir cuáles incluir en
Random Forest, calculo la matriz de correlación entre ellas para confirmar
la redundancia y elegir 1-2 representantes de este cluster en vez de las 6.

In [4]:
activity_cluster = ["active_days_obs", "out_count_obs", "num_merchant_categories_obs",
                     "total_amount_obs", "num_countries_visited_obs", "in_count_obs",
                     "avg_transaction_amount_obs", "declined_rate_obs"]

print(model_df[activity_cluster].corr().round(2))

                             active_days_obs  out_count_obs  \
active_days_obs                         1.00           0.87   
out_count_obs                           0.87           1.00   
num_merchant_categories_obs             0.85           0.80   
total_amount_obs                        0.10           0.10   
num_countries_visited_obs               0.65           0.56   
in_count_obs                            0.66           0.60   
avg_transaction_amount_obs             -0.00          -0.00   
declined_rate_obs                       0.09           0.05   

                             num_merchant_categories_obs  total_amount_obs  \
active_days_obs                                     0.85              0.10   
out_count_obs                                       0.80              0.10   
num_merchant_categories_obs                         1.00              0.08   
total_amount_obs                                    0.08              1.00   
num_countries_visited_obs                 

In [5]:
activity_cluster = ["active_days_obs", "out_count_obs", "num_merchant_categories_obs",
                     "total_amount_obs", "num_countries_visited_obs", "in_count_obs",
                     "avg_transaction_amount_obs", "declined_rate_obs"]

print(model_df[activity_cluster].corr().round(2))

                             active_days_obs  out_count_obs  \
active_days_obs                         1.00           0.87   
out_count_obs                           0.87           1.00   
num_merchant_categories_obs             0.85           0.80   
total_amount_obs                        0.10           0.10   
num_countries_visited_obs               0.65           0.56   
in_count_obs                            0.66           0.60   
avg_transaction_amount_obs             -0.00          -0.00   
declined_rate_obs                       0.09           0.05   

                             num_merchant_categories_obs  total_amount_obs  \
active_days_obs                                     0.85              0.10   
out_count_obs                                       0.80              0.10   
num_merchant_categories_obs                         1.00              0.08   
total_amount_obs                                    0.08              1.00   
num_countries_visited_obs                 

## Feature set final: dos dimensiones de actividad, no una

La correlación confirma que "frecuencia de uso" (active_days_obs y sus 4
correlacionadas) y "monto movido" (total_amount_obs) son dos dimensiones
genuinamente distintas del comportamiento, no la misma señal repetida --
correlación entre ambos grupos es de solo 0.06-0.12. Me quedo con
active_days_obs como representante del cluster de frecuencia (ya justifiqué
antes por qué es más robusta a ráfagas que num_transactions), y con
total_amount_obs como la dimensión de monto. Descarto out_count_obs,
num_merchant_categories_obs, in_count_obs y num_countries_visited_obs por
redundancia con active_days_obs -- no aportan información nueva una vez que
ya tengo la frecuencia, y reducir el feature set ayuda a que SHAP reparta
el crédito de forma más interpretable en vez de diluirlo entre 5 variables
casi idénticas.

In [6]:
print(model_df.groupby("user_settings_crypto_unlocked")["churn_model"].agg(["mean", "count"]))

                                   mean  count
user_settings_crypto_unlocked                 
0                              0.188032  15859
1                              0.065831   3509
